# **INTRO**

As an initial step in the project, we conducted a focused exploratory data analysis of the LEAF prompt dataset in order to understand its structure, quality, and semantic characteristics before making any embedding-related design choices. The dataset contains 20,000 records and 20 fields, with no missing values, which makes it suitable for a robust semantic retrieval pipeline. Our analysis examined the distribution of languages, categories, subcategories, difficulty levels, text lengths, placeholders, and tags, as well as the distinction between semantic fields and metadata. This preliminary stage was essential to ensure that the embedding strategy would be grounded in the actual properties of the corpus rather than selected arbitrarily.

In [ ]:
import json
import pandas as pd
import numpy as np
from collections import Counter
import re

INPUT_FILE = "/content/dataset.json"

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Total records:", len(data))
print("Data structure type:", type(data))
print("First record keys:", list(data[0].keys()))

In [ ]:
df = pd.DataFrame(data)
print("Dataset shape:", df.shape)
df.head(3)
print("DATA TYPES:")
print(df.dtypes)
print()

print("MISSING VALUES:")
print(df.isna().sum().sort_values(ascending=False))

**Dataset overview**

The dataset contains **20,000 prompt records** and **20 fields**, providing a sufficiently large corpus for building and testing a semantic retrieval pipeline. The absence of missing values across all columns is particularly valuable, as it removes the need for imputation or record filtering before the embedding stage. The schema also shows a clear separation between textual, categorical, and behavioral metadata, which supports a modular design for downstream retrieval and rerankin

In [ ]:
print("LANGUAGE DISTRIBUTION:")
print(df["language"].value_counts())
print()

print("TOP 20 CATEGORIES:")
print(df["category"].value_counts().head(20))
print()

print("TOP 20 SUBCATEGORIES:")
print(df["subcategory"].value_counts().head(20))
print()

print("DIFFICULTY DISTRIBUTION:")
print(df["difficulty"].value_counts())
print()

print("HAS_PLACEHOLDERS DISTRIBUTION:")
print(df["has_placeholders"].value_counts())

**Core distributions for embeddings**

**Language distribution**

The corpus is overwhelmingly **English-dominant** *(19,991 out of 20,000 records)*, with only a negligible number of prompts in other languages. This strongly supports the use of an English-focused embedding model, while suggesting that multilingual handling can be treated as an edge case rather than a primary design requirement.

**Category distribution**

The category distribution reveals a broad thematic spread, with strong concentration in domains such as creative writing, marketing, sales, legal, and customer support. This variety is important because the embedding space must capture meaning across multiple domains rather than within a single narrow topic. At the same time, the presence of dominant categories may bias retrieval quality if evaluation queries are not selected carefully across the corpus.

**Subcategory distribution**

Subcategories appear more fine-grained and operationally informative than top-level categories. Labels such as ticket-responses, contract-drafting, cold-outreach, and plot-structure suggest that the dataset is not only topically diverse but also organized around task intent, which may be beneficial for semantic disambiguation when prompts are short or underspecified.

**Difficulty distribution**

The dataset is centered around intermediate-level prompts, followed by beginner, advanced, and expert prompts. This indicates that the corpus mostly represents practical, applied requests rather than highly specialized expert-only prompts. For embeddings, this matters because semantic variation is likely driven more by task formulation and domain vocabulary than by technical depth alone.

**Placeholder distribution**

Most prompts **do not contain placeholders**, while a meaningful minority does. This suggests that placeholders are not universal enough to define the corpus, but they are common enough to deserve explicit consideration during preprocessing. The embedding pipeline should therefore remain robust to both fully specified prompts and reusable template-style prompt

In [ ]:
df["title_len_chars"] = df["title"].fillna("").astype(str).str.len()
df["content_len_chars"] = df["content"].fillna("").astype(str).str.len()

df["title_len_words"] = df["title"].fillna("").astype(str).str.split().str.len()
df["content_len_words"] = df["content"].fillna("").astype(str).str.split().str.len()

print("=== TEXT LENGTH STATS ===")
print(df[[
    "title_len_chars", "content_len_chars",
    "title_len_words", "content_len_words"
]].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

**Text length analysis**

The length statistics show a clear asymmetry between the two main textual fields: ***titles are generally short***, while **contents are** substantially **longer** and more informative. On average, **titles contain roughly 4.4 word**s, whereas **contents contain about 54.7 words**, with a long upper tail extending beyond 200 words. This confirms that content is the main carrier of semantic information, while title functions more as a compressed cue than a complete description. Extremely short prompts and very long prompts coexist in the same corpus, which may create heterogeneity in embedding quality and should be considered during model selection and evaluation

In [ ]:
def is_very_short(text, threshold=4):
    return len(str(text).split()) <= threshold

def has_all_caps(text):
    s = str(text)
    letters = [c for c in s if c.isalpha()]
    if not letters:
        return False
    upper_ratio = sum(c.isupper() for c in letters) / len(letters)
    return upper_ratio > 0.7

def has_informal_style(text):
    s = str(text).lower()
    patterns = [
        r"\bidk\b", r"\bwats\b", r"\bpls\b", r"\bwut\b", r"\bim\b",
        r"\bcuz\b", r"\btho\b", r"\bgimme\b", r"\babt\b", r"\blol\b"
    ]
    return any(re.search(p, s) for p in patterns)

df["title_very_short"] = df["title"].apply(is_very_short)
df["title_all_caps_style"] = df["title"].apply(has_all_caps)
df["title_informal_style"] = df["title"].apply(has_informal_style)

print("TITLE NOISE CHECK:")
print("Very short titles (%):", round(df["title_very_short"].mean() * 100, 2))
print("All-caps style titles (%):", round(df["title_all_caps_style"].mean() * 100, 2))
print("Informal-style titles (%):", round(df["title_informal_style"].mean() * 100, 2))

**Title quality / noise check**

The title analysis indicates that a large share of titles are very short (over 64%), while all-caps and strongly informal forms are present but much less frequent. This pattern suggests that titles cannot be assumed to be consistently descriptive: in many cases they provide useful context, but in others they are vague, colloquial, or stylistically noisy. As a result, titles may enrich the embedding input, but they should not be treated as a reliable standalone semantic field.

In [ ]:
df["n_placeholders"] = df["placeholders"].apply(lambda x: len(x) if isinstance(x, list) else 0)

print("PLACEHOLDER STATS:")
print(df["n_placeholders"].describe())
print()

all_placeholders = Counter()
for row in df["placeholders"]:
    if isinstance(row, list):
        all_placeholders.update(row)

print("TOP 30 PLACEHOLDERS:")
for ph, cnt in all_placeholders.most_common(30):
    print(f"{ph}: {cnt}")

**Placeholder analysis**

The placeholder statistics show that the median prompt contains no placeholders, but the maximum reaches 10, confirming that some prompts are highly templated. The most frequent placeholders — such as industry, company_name, topic, product_name, and audience — capture domain-specific variables rather than arbitrary syntax. This means placeholders may preserve useful structural intent, especially in reusable business and professional prompts. Their treatment should therefore be cautious: removing them entirely could discard part of the prompt’s functional meaning.

In [ ]:
df["n_tags"] = df["tags"].apply(lambda x: len(x) if isinstance(x, list) else 0)

print("TAG COUNT STATS:")
print(df["n_tags"].describe())
print()

all_tags = Counter()
for row in df["tags"]:
    if isinstance(row, list):
        all_tags.update(row)

print("TOP 50 TAGS:")
for tag, cnt in all_tags.most_common(50):
    print(f"{tag}: {cnt}")

**Tag analysis**

Tags are relatively dense and well populated, with a median of **4 tags per prompt** and an upper bound of 7. The most frequent tags indicate recurring themes such as **templates, compliance, documentation, analysis, email, Python, and marketing**. This suggests that tags provide concise semantic descriptors that may complement the main text, particularly when the prompt itself is brief or underspecified. At the same time, tags are editorial metadata, so their contribution should be validated empirically rather than assumed to improve retrieval by default.

In [ ]:
print("5 SHORTEST PROMPTS BY WORD COUNT:")
shortest = df.sort_values("content_len_words").head(5)
for _, row in shortest.iterrows():
    print("\nID:", row["id"])
    print("Title:", row["title"])
    print("Content:", row["content"])
    print("Category:", row["category"], "| Subcategory:", row["subcategory"])
    print("-" * 80)

print("\n 5 LONGEST PROMPTS BY WORD COUNT:")
longest = df.sort_values("content_len_words", ascending=False).head(5)
for _, row in longest.iterrows():
    print("\nID:", row["id"])
    print("Title:", row["title"])
    print("Content preview:", row["content"][:1200])
    print("Category:", row["category"], "| Subcategory:", row["subcategory"])
    print("-" * 80)

**Shortest prompts by word count**

The shortest prompts illustrate a critical challenge for the embedding stage: some records contain extremely limited lexical information, such as one-word or near one-word requests. In these cases, semantic meaning is only partially expressed in the content field and is more likely to emerge from the combination of title, category, subcategory, and tags. These examples show why an embedding strategy based on content alone may underperform on a non-trivial subset of the datase

**Longest prompts by word count**

The longest prompts resemble full user problem descriptions rather than compact prompt templates. They contain narrative context, uncertainty, and implicit goals, which provide rich semantic material for dense representations. These cases are likely to benefit strongly from modern sentence-level or text-level embeddings, since the main challenge is not lexical sparsity but preserving topical and functional coherence across a longer sequence

In [ ]:
print("5 PROMPTS WITH PLACEHOLDERS:")
with_ph = df[df["has_placeholders"] == True].head(5)

for _, row in with_ph.iterrows():
    print("\nID:", row["id"])
    print("Title:", row["title"])
    print("Placeholders:", row["placeholders"])
    print("Content:", row["content"][:1000])
    print("-" * 80)

In [ ]:
semantic_fields = [
    "title", "content", "category", "subcategory", "tags", "placeholders", "language"
]

metadata_fields = [
    "author_reputation", "version", "fork_count", "likes", "upvotes", "downvotes",
    "views", "uses", "created_at", "has_placeholders", "difficulty", "target_model"
]

print("SEMANTIC FIELDS:")
for c in semantic_fields:
    print("-", c)

print("\n METADATA / RANKING / FILTERING FIELDS:")
for c in metadata_fields:
    print("-", c)

**Semantic vs metadata fields**

The distinction between semantic fields and metadata fields is well justified by the dataset design. Fields such as content, title, category, subcategory, tags, and placeholders contribute directly to meaning or task intent, whereas engagement-related and behavioral variables such as likes, views, uses, and reputation are better interpreted as ranking or filtering signals. This separation supports a clean architecture in which semantic similarity is modeled independently from popularity or platform performance

**Final Overwiev**


The exploratory analysis highlighted several findings that are directly relevant to the embedding component. First, the corpus is overwhelmingly English-dominant, which supports the adoption of an English-oriented embedding model. Second, the content field emerges as the main semantic carrier, both by dataset definition and by observed text-length patterns, whereas title appears too short and inconsistent to be used as a standalone semantic representation. At the same time, categories, subcategories, and tags may provide useful contextual support, especially for short or underspecified prompts. Finally, engagement-related variables such as likes, upvotes, views, uses, and author reputation should not be incorporated into the semantic embedding itself, since they reflect behavioral or ranking signals rather than prompt meaning. These observations provide a principled basis for designing the embedding input and for keeping semantic representation separate from later ranking and reranking stages

# **Definition and Construction of Embedding Input Configurations**

**Configuration design**

To support the embedding stage, we defined two alternative textual input configurations. The first configuration **(Config A)** uses only the content field, since the dataset documentation explicitly identifies it as the primary semantic field. This configuration serves as the most direct semantic baseline.

The second configuration **(Config B)** augments the main textual content with additional structured context, namely title, category, subcategory, and up to four tags, followed by the content itself. This design was motivated by the exploratory analysis, which showed that several prompts are short, underspecified, or stylistically noisy, making contextual metadata potentially useful for disambiguation.

At the same time, the number of tags included was capped at four in order to preserve semantic signal while limiting the introduction of excessive editorial noise, additionally the number of tags included in Config B was capped at four, as this value is closely aligned with the average tag count observed in the dataset. This choice was intended to retain a representative amount of contextual information while preventing the input from becoming excessively influenced by auxiliary editorial metadata

If fewer than four tags were available, only the existing tags were included. If no tags were present, the tag line was omitted entirely.

In [ ]:
import re


def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def format_tags(tags, max_tags=4):
    if not isinstance(tags, list):
        return ""
    tags = [clean_text(t) for t in tags if clean_text(t)]
    return ", ".join(tags[:max_tags])

def build_config_a(row):
    content = clean_text(row.get("content", ""))
    return content

def build_config_b(row):
    title = clean_text(row.get("title", ""))
    category = clean_text(row.get("category", ""))
    subcategory = clean_text(row.get("subcategory", ""))
    tags = format_tags(row.get("tags", []), max_tags=4)
    content = clean_text(row.get("content", ""))

    parts = []

    if title:
        parts.append(f"Title: {title}")
    if category:
        parts.append(f"Category: {category}")
    if subcategory:
        parts.append(f"Subcategory: {subcategory}")
    if tags:
        parts.append(f"Tags: {tags}")
    if content:
        parts.append(f"Content: {content}")

    return "\n".join(parts)

df["text_config_a"] = df.apply(build_config_a, axis=1)
df["text_config_b"] = df.apply(build_config_b, axis=1)

print("Config A example:\n")
print(df["text_config_a"].iloc[0][:1000])

print("\n" + "="*80 + "\n")

print("Config B example:\n")
print(df["text_config_b"].iloc[0][:1000])

Before generating embeddings, a light preprocessing strategy was applied to standardize the textual input while preserving the original semantic content of each prompt. Preprocessing was limited to structural normalization steps, such as whitespace cleanup, safe handling of empty fields, and controlled formatting of metadata fields included in the embedding input. No aggressive normalization was applied: typos, informal phrasing, capitalization patterns, punctuation, and template placeholders were preserved in order to retain the natural characteristics of user-generated prompts.

# **Light Preprocessing Strategy for Embedding Input Construction**

The preprocessing operations used for input construction were implemented through the `clean_text()` and `format_tags()` functions introduced in the previous section. In this stage, we focus on the rationale behind this conservative preprocessing design rather than repeating the implementation details

Following the definition of the embedding input configurations, a light preprocessing strategy was adopted to standardize the textual input while preserving the semantic characteristics of the original prompts. Given the user-generated nature of the dataset, the objective was not to aggressively normalize the text, but rather to ensure a clean and consistent input format for embedding generation. This choice was particularly important because the corpus includes short prompts, informal phrasing, abbreviations, spelling noise, and template-based prompts, all of which may still carry task-relevant information in a semantic retrieval setting.

**Preprocessing was therefore limited to structural normalization steps only.**
   In particular, template placeholders were preserved rather than removed, since they may encode functional prompt structure and variable task intent. Although placeholders were not introduced as a separate dedicated field in the current input configurations, retaining them within the original prompt text allowed the embedding stage to preserve this potentially meaningful structural signal. Similarly, punctuation, informal expressions, abbreviations, and spelling noise were maintained rather than aggressively normalized. While such features may appear noisy from a traditional text-cleaning perspective, they are part of the natural formulation of user-generated prompts and may still contribute to the semantic profile captured by the embedding model. Removing them systematically would risk oversimplifying the text and discarding potentially relevant cues related to tone, structure, or task framing.

**Preprocessing operations applied**

*   whitespace normalization
*   safe conversion of textual fields to string format
*   safe handling of empty values
*   controlled formatting of tag lists

**Preprocessing operations intentionally avoided**

*   spelling correction
*   lowercasing of the full prompt text
*   punctuation removal
*   placeholder removal
*   content rewriting or paraphrasing

This conservative approach was intended to avoid altering the lexical form of the prompts in ways that could remove potentially useful semantic cues for the embedding model.

In [ ]:
sample_idx = [0, 10, 25, 100, 250, 500]

for i in sample_idx:
    print(f"\nROW {i}")
    print("CONFIG A:\n", df["text_config_a"].iloc[i][:1000])
    print("\nCONFIG B:\n", df["text_config_b"].iloc[i][:1000])
    print("\n" + "="*100)

**Manual Inspection of Constructed Inputs**

Before proceeding to embedding generation, a small qualitative sanity check was performed on selected prompts. This manual inspection was used to verify that the two configurations were correctly constructed, that the preprocessing steps did not distort the original prompt semantics, and that both short and long prompts remained well formatted after input constructio

# **Initial Embedding Model Selection and Baseline Representation Generation**

**Intro**

After defining the two textual input configurations and constructing the corresponding embedding inputs, the next step is to generate dense vector representations for the prompt corpus. At this stage, the goal is not yet to identify the final best-performing embedding architecture, but to establish a reliable baseline for comparing the two input configurations under the same model conditions. For this reason, the first embedding model is used as a controlled reference point, allowing the evaluation to focus primarily on the effect of input design rather than on differences between embedding models.

**Model choice**

As an initial embedding model, we selected ` sentence-transformers/all-MiniLM-L6-v2` . This model was chosen as a strong baseline for three main reasons. First, it is lightweight and computationally efficient, which makes it well suited for rapid experimentation in a Colab-based workflow. Second, it is widely used as a reliable general-purpose sentence embedding model, making it an appropriate starting point for semantic similarity tasks. Third, the dataset is overwhelmingly English-dominant, so an English-oriented sentence embedding model is a suitable match for the corpus characteristics.

At this stage, the choice of MiniLM should be interpreted as a baseline model selection, not as a final commitment. The purpose is to generate embeddings for both Config A and Config B under identical model conditions and observe whether the inclusion of structured contextual metadata improves the semantic representation of prompts. If this first comparison yields meaningful differences, a second embedding model can later be introduced in order to determine whether the observed behavior is robust across architectures rather than specific to the baseline model alone.

***The model is not assumed to be the final best option in advance***. Rather, it is selected because it offers a practical balance between semantic quality, implementation simplicity, and computational efficiency. Its usefulness will be assessed empirically through the downstream comparison of retrieval behavior across the two input configurations. In other words, the present step does not “prove” that MiniLM is the best model overall; instead, it provides a stable and interpretable first benchmark from which more informed comparisons can later be made.


A baseline embedding model was then instantiated in order to generate dense vector representations for both textual configurations. The same model was applied to Config A and Config B so that any downstream difference could be attributed, in the first instance, to the structure of the embedding input rather than to the embedding architecture itself. The resulting vectors were stored for later comparison and integration with the retrieval component

In [ ]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer
import numpy as np

model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(model_name)

print("Loaded model:", model_name)

In [ ]:
texts_a = df["text_config_a"].tolist()
texts_b = df["text_config_b"].tolist()

embeddings_a = embedding_model.encode(
    texts_a,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

embeddings_b = embedding_model.encode(
    texts_b,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embeddings A shape:", embeddings_a.shape)
print("Embeddings B shape:", embeddings_b.shape)

In [ ]:
np.save("/content/embeddings_config_a_minilm.npy", embeddings_a)
np.save("/content/embeddings_config_b_minilm.npy", embeddings_b)

print("Saved:")
print("/content/embeddings_config_a_minilm.npy")
print("/content/embeddings_config_b_minilm.npy")

In [ ]:
embedding_metadata = df[[
    "id", "title", "category", "subcategory", "tags", "content",
    "text_config_a", "text_config_b"
]].copy()

embedding_metadata.to_csv("/content/embedding_metadata_minilm.csv", index=False)

print("Saved metadata file:")
print("/content/embedding_metadata_minilm.csv")

**result comment**

The baseline embedding generation step was completed successfully using sentence-transformers/all-MiniLM-L6-v2. Dense vector representations were produced for both Config A and Config B, yielding matrices of shape (20000, 384) in both cases. This confirms that the full corpus was encoded without data loss and that the two configurations are directly comparable under identical embedding dimensionality.

As expected, the enriched configuration required longer encoding time than the baseline configuration, reflecting the additional textual context included in Config B. This difference is relevant not only from a computational perspective, but also for later methodological discussion, since any retrieval gain obtained from Config B should be interpreted alongside its higher encoding cost.

The resulting embeddings and aligned metadata were saved for downstream integration with the retrieval and vector database components.

# **CREATING THE VECTOR DATABASE**
Now that we have embedded the dataset into vectors, we can proceed to build the vectorial database for a fast and efficient retrieval of our contents.
We have decided to build 2 databases: one with only config A and one with only config B.

We do so as to be able to play around with all possible combinations of data to find the best approach when we will do operations like retrieval through cosine similarity, or retrieval based on metadata.

We will use ChromaDB and Qdrandt and based on performance we will decide which is best.

In [ ]:
!pip install chromadb
!pip install

In [ ]:
import numpy as np
import pandas as pd
import chromadb

config_a = np.load("/content/embeddings_config_a_minilm.npy")
config_b = np.load("/content/embeddings_config_b_minilm.npy")
metadata = pd.read_csv("/content/embedding_metadata_minilm.csv")
print(f"Config A: {len(config_a)}")
print(f"Config B: {len(config_b)}")
print(f"CSV rows: {len(metadata)}")

In [ ]:
chromaDB = chromadb.PersistentClient(path="/content")

In [ ]:
config_a_database = chromaDB.create_collection(name="config_a")

In [ ]:
config_b_database = chromaDB.create_collection(name="config_b")

In [ ]:
def database_adder(database, np_array, metadata, batch_size):
    vectors = np_array.tolist()
    total = len(np_array)

    for start in range(0, total, batch_size):
        end = min(start + batch_size, total)

        ids = [str(metadata["id"][i]) for i in range(start, end)]
        embeddings = [vectors[i] for i in range(start, end)]
        documents  = [str(metadata["content"][i]) for i in range(start, end)]
        metadatas = [{
            "title":       str(metadata["title"][i]),
            "category":    str(metadata["category"][i]),
            "subcategory": str(metadata["subcategory"][i]),
            "tags":        str(metadata["tags"][i]),
        } for i in range(start, end)]

        database.add(
            ids=ids,
            documents=documents,
            embeddings=embeddings,
            metadatas=metadatas,
        )

In [ ]:
database_adder(config_a_database, config_a, metadata, 100)

In [ ]:
config_a_database.get(ids="pk_03138", include=["embeddings"])

In [ ]:
database_adder(config_b_database, config_b, metadata, 100)

In [ ]:
config_b_database.get(ids="pk_03138")